# Required EDA and Preprocessing — ALPR Dataset

This notebook provides a compact, runnable workflow for exploratory data analysis (EDA) and the preprocessing stages required by the pipeline: inspection, harmonization, configurable preprocessing, and splitting.
Use this as a canonical, repeatable script to run on a workspace to produce EDA figures and preprocessed data suitable for training.

In [5]:
from __future__ import annotations
import sys
from pathlib import Path

def _find_project_root(marker="pyproject.toml"):
    path = Path.cwd().resolve()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    return path

PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 120

print(f"Project root: {PROJECT_ROOT}")
import platform
print(f"Python: {platform.python_version()}")

Project root: C:\Users\Admin\Documents\GitHub\AI-Tools-Project
Python: 3.12.10


In [6]:
# Core imports for EDA + preprocessing
from alpr_dataset.config import PipelineConfig
from alpr_dataset.logging_setup import setup_logging
from alpr_dataset.io_utils import list_images, safe_read_image
from alpr_dataset.annotations.loader import load_dataset_annotations
from alpr_dataset.inspection.image_stats import batch_compute_stats
from alpr_dataset.inspection.hashing import find_duplicates
from alpr_dataset.eda.quality import build_quality_report
from alpr_dataset.harmonization.harmonizer import harmonize_dataset
from alpr_dataset.preprocessing.pipeline import PreprocessingPipeline, STEP_REGISTRY
from alpr_dataset.splitting.splitter import stratified_split, write_split_manifests

# Load configuration files
config = PipelineConfig.load(
    PROJECT_ROOT / "configs" / "pipeline_config.yaml",
    PROJECT_ROOT / "configs" / "datasets.yaml",
)
prep_config = config.preprocessing_config(PROJECT_ROOT / "configs" / "preprocessing_config.yaml")
split_cfg = config.split_config(PROJECT_ROOT / "configs" / "preprocessing_config.yaml")
logger = setup_logging(config.logs_dir, name="alpr_dataset")

print(f"Datasets: {[s.name for s in config.datasets]}")
print(f"Reports dir: {config.reports_dir}")
print(f"Processed dir: {config.data_processed_dir}")

Datasets: ['dataset_A', 'dataset_B']
Reports dir: C:\Users\Admin\Documents\GitHub\AI-Tools-Project\reports
Processed dir: C:\Users\Admin\Documents\GitHub\AI-Tools-Project\data\processed


In [7]:
# 1) Quick dataset scan (counts + formats)
scan_summary = {}
for spec in config.datasets:
    imgs = list_images(spec.root)
    scan_summary[spec.name] = dict(n_images=len(imgs), sample=imgs[:3])
    print(f"{spec.name}: {len(imgs)} images (example: {[(p.name) for p in imgs[:3]]})")

dataset_A: 464 images (example: ['a8b8f6be161a4bdcabcc947b3e72f8b2.jpg', 'Capture.JPG', 'IMG20221107210304.jpg'])
dataset_B: 2087 images (example: ['0001.jpg', '0002.jpg', '0003.jpg'])


In [8]:
# 2) Load annotations for every dataset
all_annotations = {}
for spec in config.datasets:
    ann = load_dataset_annotations(spec)
    all_annotations[spec.name] = ann
    print(f"{spec.name}: {len(ann)} annotated images, {sum(a.n_boxes for a in ann)} boxes")

dataset_A: 231 annotated images, 272 boxes


[07/19/26 19:10:20] WARNING  Skipping unreadable image for YOLO annotation:                                        
                             C:\Users\Admin\Documents\GitHub\AI-Tools-Project\data\raw\dataset_B\Vehicles\2018.jpg

dataset_B: 2086 annotated images, 2142 boxes


In [9]:
# 3) Compute per-image statistics and find near-duplicates
all_data = {}
for spec in config.datasets:
    images = list_images(spec.root)
    stats = batch_compute_stats(images)
    dupes = find_duplicates(images, hamming_threshold=config.duplicate_hash_threshold).near_duplicates
    all_data[spec.name] = {"images": images, "stats": stats, "dupes": dupes}
    valid = [s for s in stats if not s.is_corrupted]
    print(f"{spec.name}: {len(images)} images, {len(valid)} valid, {sum(1 for s in stats if s.is_corrupted)} corrupted")

dataset_A: 464 images, 464 valid, 0 corrupted


[07/19/26 19:14:41] WARNING  Could not compute phash for                                                           
                             C:\Users\Admin\Documents\GitHub\AI-Tools-Project\data\raw\dataset_B\Vehicles\2018.jpg:
                             image file is truncated (5 bytes not processed)

dataset_B: 2087 images, 2086 valid, 1 corrupted


In [10]:
# 4) Generate quick EDA plots for a chosen dataset (first one)
spec = config.datasets[0]
d = all_data[spec.name]
valid = [s for s in d['stats'] if not s.is_corrupted]
widths = [s.width for s in valid]
heights = [s.height for s in valid]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=40, color='#3d5a80', edgecolor='white')
axes[0].set_xlabel('Width (px)'); axes[0].set_ylabel('Frequency'); axes[0].set_title(f'{spec.name}: Width')
axes[1].hist(heights, bins=40, color='#ee6c4d', edgecolor='white')
axes[1].set_xlabel('Height (px)'); axes[1].set_ylabel('Frequency'); axes[1].set_title(f'{spec.name}: Height')
fig.tight_layout(); plt.show()

C:\Users\Admin\AppData\Local\Temp\ipykernel_44232\1224464877.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); plt.show()


In [11]:
# 5) Build quality reports (JSON + MD) for each dataset
for spec in config.datasets:
    image_paths = list_images(spec.root)
    report = build_quality_report(spec.name, all_annotations[spec.name], image_paths, config.reports_dir / 'quality', hamming_threshold=config.duplicate_hash_threshold,
        blur_threshold=config.blur_threshold)
    print(f"Wrote quality report for {spec.name}: {config.reports_dir / 'quality'}")

[07/19/26 19:17:53] INFO     Wrote quality report JSON ->                                                          
                             C:\Users\Admin\Documents\GitHub\AI-Tools-Project\reports\quality\dataset_A_quality_rep
                             ort.json

                    INFO     Wrote quality report Markdown ->                                                      
                             C:\Users\Admin\Documents\GitHub\AI-Tools-Project\reports\quality\dataset_A_quality_rep
                             ort.md

Wrote quality report for dataset_A: C:\Users\Admin\Documents\GitHub\AI-Tools-Project\reports\quality


[07/19/26 19:19:31] WARNING  Could not compute phash for                                                           
                             C:\Users\Admin\Documents\GitHub\AI-Tools-Project\data\raw\dataset_B\Vehicles\2018.jpg:
                             image file is truncated (5 bytes not processed)

[07/19/26 19:19:39] INFO     Wrote quality report JSON ->                                                          
                             C:\Users\Admin\Documents\GitHub\AI-Tools-Project\reports\quality\dataset_B_quality_rep
                             ort.json

                    INFO     Wrote quality report Markdown ->                                                      
                             C:\Users\Admin\Documents\GitHub\AI-Tools-Project\reports\quality\dataset_B_quality_rep
                             ort.md

Wrote quality report for dataset_B: C:\Users\Admin\Documents\GitHub\AI-Tools-Project\reports\quality


In [12]:
# 6) Harmonize datasets into unified YOLO layout
unified_root = config.data_processed_dir / 'unified'
unified_root.mkdir(parents=True, exist_ok=True)
for spec in config.datasets:
    records = harmonize_dataset(
        dataset_name=spec.name,
        annotations=all_annotations[spec.name],
        unified_class_map=config.unified_class_map,
        local_class_map=spec.class_map or {},
        output_root=unified_root,
    )
    print(f"{spec.name}: harmonized {len(records)} files -> {unified_root}")

dataset_A: harmonized 231 files -> C:\Users\Admin\Documents\GitHub\AI-Tools-Project\data\processed\unified


dataset_B: harmonized 2086 files -> C:\Users\Admin\Documents\GitHub\AI-Tools-Project\data\processed\unified


In [13]:
# 7) Configure and run preprocessing pipeline (writes preprocessed images)
pipeline = PreprocessingPipeline(prep_config)
preprocessed_root = config.data_processed_dir / 'preprocessed'
preprocessed_root.mkdir(parents=True, exist_ok=True)
for spec in config.datasets:
    imgs = list_images(spec.root)
    out = pipeline.run_on_dataset(imgs, preprocessed_root / spec.name / 'images', spec.name, comparisons_dir=config.reports_dir / 'before_after' / spec.name)
    print(f"{spec.name}: preprocessed {len(out)} images -> {preprocessed_root / spec.name / 'images'}")

[07/19/26 19:41:09] INFO     Preprocessed 464/464 images for 'dataset_A' ->                                        
                             C:\Users\Admin\Documents\GitHub\AI-Tools-Project\data\processed\preprocessed\dataset_A
                             \images

dataset_A: preprocessed 464 images -> C:\Users\Admin\Documents\GitHub\AI-Tools-Project\data\processed\preprocessed\dataset_A\images


[dataset_B] preprocessing:  97%|█████████▋| 2017/2087 [10:01<00:20,  3.47it/s]

[07/19/26 19:51:11] WARNING  Skipping unreadable image during preprocessing:                                       
                             C:\Users\Admin\Documents\GitHub\AI-Tools-Project\data\raw\dataset_B\Vehicles\2018.jpg

[07/19/26 19:51:29] INFO     Preprocessed 2086/2087 images for 'dataset_B' ->                                      
                             C:\Users\Admin\Documents\GitHub\AI-Tools-Project\data\processed\preprocessed\dataset_B
                             \images

dataset_B: preprocessed 2086 images -> C:\Users\Admin\Documents\GitHub\AI-Tools-Project\data\processed\preprocessed\dataset_B\images


In [14]:
# 8) Stratified split of preprocessed dataset (manifests)
for spec in config.datasets:
    annotations = load_dataset_annotations(spec)
    result = stratified_split(annotations, split_cfg)
    out_dir = config.data_processed_dir / 'split' / spec.name
    write_split_manifests(result, out_dir)
    print(f"Wrote split manifests -> {out_dir}")

[07/19/26 19:51:43] INFO     Wrote split manifests ->                                                              
                             C:\Users\Admin\Documents\GitHub\AI-Tools-Project\data\processed\split\dataset_A       
                             ({'n_train': 162, 'n_val': 35, 'n_test': 34, 'n_total': 231})

Wrote split manifests -> C:\Users\Admin\Documents\GitHub\AI-Tools-Project\data\processed\split\dataset_A


[07/19/26 19:51:54] WARNING  Skipping unreadable image for YOLO annotation:                                        
                             C:\Users\Admin\Documents\GitHub\AI-Tools-Project\data\raw\dataset_B\Vehicles\2018.jpg

[07/19/26 19:51:55] INFO     Wrote split manifests ->                                                              
                             C:\Users\Admin\Documents\GitHub\AI-Tools-Project\data\processed\split\dataset_B       
                             ({'n_train': 1460, 'n_val': 313, 'n_test': 313, 'n_total': 2086})

Wrote split manifests -> C:\Users\Admin\Documents\GitHub\AI-Tools-Project\data\processed\split\dataset_B


## Summary
- EDA: dataset scanning, per-image stats, duplicate detection, and quick plots are included.
- Harmonization: converts heterogeneous annotations to unified YOLO layout under `data/processed/unified/`.
- Preprocessing: configurable pipeline applied and saved to `data/processed/preprocessed/`.
- Splitting: stratified manifests are written to `data/processed/split/`.

Next: run the notebook (cell-by-cell) to produce outputs; if you want, I can run the notebook now and report results.